# 🧠 Model Training Pipeline - HarmonyMind
This notebook contains the complete text emotion detection training pipeline using a **Bidirectional LSTM** model, starting from raw text files, performing tokenization, padding, and optimization callbacks.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau


## Resolve Paths and Load Train/Val Sets


In [ ]:
project_root = os.path.dirname(os.getcwd())
train_path = os.path.join(project_root, 'datasets', 'train.txt')
val_path = os.path.join(project_root, 'datasets', 'val.txt')

df_train = pd.read_csv(train_path, sep=';', header=None, names=['text', 'emotion'])
df_val = pd.read_csv(val_path, sep=';', header=None, names=['text', 'emotion'])
print(f'Train shape: {df_train.shape}, Val shape: {df_val.shape}')


## Tokenization & Padding Sequence Preparation


In [ ]:
MAX_WORDS = 10000
MAX_LEN = 100

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df_train['text'])

X_train = pad_sequences(tokenizer.texts_to_sequences(df_train['text']), maxlen=MAX_LEN)
X_val = pad_sequences(tokenizer.texts_to_sequences(df_val['text']), maxlen=MAX_LEN)

le = LabelEncoder()
y_train = le.fit_transform(df_train['emotion'])
y_val = le.transform(df_val['emotion'])
num_classes = len(le.classes_)
print('Classes:', list(le.classes_))


## Model Architecture Design (Bidirectional LSTM)


In [ ]:
model = Sequential([
    Embedding(MAX_WORDS, 64, input_shape=(MAX_LEN,)),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


## Setup Callbacks & Run Model Fitting


In [ ]:
best_model_path = os.path.join(project_root, 'models', 'best_model.h5')
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint(filepath=best_model_path, monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.0001)
]

# Train for 5 epochs
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


## Save Tokenizer and Label Encoder


In [ ]:
with open(os.path.join(project_root, 'models', 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)
with open(os.path.join(project_root, 'models', 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)

print('Training script run complete!')
